In [7]:
import pandas as pd

arquivo = "04_PENSE2024_SEM_NAN.parquet"

df = pd.read_parquet(arquivo)

print(df.dtypes)

REGIAO - código de regiões                                                                                                                                                                                                                                   int64
UF - código UF                                                                                                                                                                                                                                               int64
CAPITAL - indica se é município da capital                                                                                                                                                                                                                   int64
B01001A - Qual é o seu sexo?                                                                                                                                                                                                   

In [1]:
import pandas as pd
import re

# ============================================================
# 1. ARQUIVOS
# ============================================================

arquivo_parquet = "04_PENSE2024_SEM_NAN.parquet"
arquivo_excel = "01_Colunas_recomendadas_pense_2024_final.xlsx"
arquivo_saida = "05_PENSE2024_NUMERACAO_NOVA.parquet"


# ============================================================
# 2. LER OS ARQUIVOS
# ============================================================

df = pd.read_parquet(arquivo_parquet)

regras = pd.read_excel(
    arquivo_excel,
    header=1
)


# ============================================================
# 3. NOMES DAS COLUNAS DO EXCEL
# ============================================================

coluna_codigo = "Código da coluna"
coluna_regra = "Nova numeracao"

if coluna_codigo not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codigo}' não encontrada no Excel."
    )

if coluna_regra not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_regra}' não encontrada no Excel."
    )


# ============================================================
# 4. FUNÇÃO PARA NORMALIZAR VALORES
# ============================================================
# O Parquet pode trazer:
#
#     1
#     1.0
#     "1"
#     "1.0"
#     " 1 "
#
# Todos passam a ser tratados como "1".
#
# Isso é importante porque as regras do Excel normalmente
# aparecem como 1, 2, -9, etc., enquanto o Parquet pode
# armazenar os mesmos códigos como 1.0, 2.0, -9.0.

def normalizar_valor(valor):

    if pd.isna(valor):
        return None

    texto = str(valor).strip()

    if texto == "":
        return None

    try:

        numero = float(texto)

        if numero.is_integer():
            return str(int(numero))

    except (ValueError, TypeError):
        pass

    return texto


# ============================================================
# 5. FUNÇÃO PARA CRIAR O MAPA DE CONVERSÃO
# ============================================================

def criar_mapa(regra):

    mapa = {}

    if pd.isna(regra):
        return mapa

    regra = str(regra).strip()

    # Remove parênteses que eventualmente apareçam no final
    regra = regra.rstrip(")")

    # Exemplos:
    #
    # 1=3,2=1,-9/-2/-1=0
    # 1/2=1,3=2,4/5=3,-9/-2/-1=0

    partes = regra.split(",")

    for parte in partes:

        parte = parte.strip()

        if not parte or "=" not in parte:
            continue

        origem, destino = parte.split("=", 1)

        origem = origem.strip()
        destino = destino.strip()

        valores_origem = origem.split("/")

        for valor in valores_origem:

            valor = valor.strip()

            if valor:

                mapa[normalizar_valor(valor)] = destino

    return mapa


# ============================================================
# 6. FUNÇÃO PARA ENCONTRAR A COLUNA NO PARQUET
# ============================================================

def encontrar_coluna(codigo, colunas_parquet):

    codigo = str(codigo).strip()

    # Primeiro: correspondência exata
    if codigo in colunas_parquet:
        return codigo

    # Segundo: código + " - "
    prefixo = codigo + " - "

    candidatas = [
        coluna
        for coluna in colunas_parquet
        if str(coluna).startswith(prefixo)
    ]

    if len(candidatas) == 1:
        return candidatas[0]

    if len(candidatas) > 1:

        print(
            f"ATENÇÃO: mais de uma coluna encontrada "
            f"para o código '{codigo}':"
        )

        for coluna in candidatas:
            print(f"    {coluna}")

        return None

    return None


# ============================================================
# 7. DIAGNÓSTICO DOS TIPOS DO PARQUET
# ============================================================

print("=" * 80)
print("TIPOS DAS COLUNAS DO PARQUET")
print("=" * 80)

print(df.dtypes.to_string())

print()
print("=" * 80)
print("EXEMPLO DOS VALORES REAIS")
print("=" * 80)

for coluna in df.columns[:10]:

    print()
    print(f"COLUNA: {coluna}")
    print(f"TIPO:   {df[coluna].dtype}")
    print(
        "VALORES:",
        df[coluna].drop_duplicates().head(10).tolist()
    )


# ============================================================
# 8. PROCESSAR TODAS AS REGRAS
# ============================================================

relatorio = []
diagnostico = []

colunas_processadas = 0
total_alteracoes = 0
total_correspondencias = 0


for _, linha in regras.iterrows():

    codigo = linha[coluna_codigo]

    if pd.isna(codigo):
        continue

    codigo = str(codigo).strip()

    regra = linha[coluna_regra]

    if pd.isna(regra):
        continue

    regra = str(regra).strip()

    if regra == "":
        continue


    # --------------------------------------------------------
    # Encontrar a coluna real
    # --------------------------------------------------------

    coluna_real = encontrar_coluna(
        codigo,
        df.columns
    )

    if coluna_real is None:

        print(
            f"AVISO: código '{codigo}' "
            f"não encontrado no Parquet."
        )

        relatorio.append({
            "codigo_excel": codigo,
            "coluna_parquet": "",
            "regra": regra,
            "status": "NÃO ENCONTRADA",
            "correspondencias": 0,
            "alteracoes": 0
        })

        continue


    # --------------------------------------------------------
    # Criar mapa
    # --------------------------------------------------------

    mapa = criar_mapa(regra)

    if not mapa:

        relatorio.append({
            "codigo_excel": codigo,
            "coluna_parquet": coluna_real,
            "regra": regra,
            "status": "REGRA VAZIA/INVÁLIDA",
            "correspondencias": 0,
            "alteracoes": 0
        })

        continue


    # --------------------------------------------------------
    # Normalizar os valores da coluna SOMENTE para a busca
    # --------------------------------------------------------
    # Não usamos replace() diretamente porque:
    #
    #     1.0 != 1
    #
    # A série normalizada transforma ambos em "1".

    valores_antes = df[coluna_real].copy()

    valores_normalizados = (
        valores_antes
        .map(normalizar_valor)
    )


    # --------------------------------------------------------
    # Identificar quais valores realmente existem
    # e quais deles possuem regra
    # --------------------------------------------------------

    frequencias = (
        valores_normalizados
        .value_counts(dropna=False)
    )

    correspondencias = 0

    for valor, quantidade in frequencias.items():

        if valor in mapa:
            correspondencias += int(quantidade)

            diagnostico.append({
                "codigo_excel": codigo,
                "coluna_parquet": coluna_real,
                "valor_encontrado": valor,
                "quantidade": int(quantidade),
                "novo_valor": mapa[valor]
            })


    # --------------------------------------------------------
    # Aplicar a transformação
    # --------------------------------------------------------

    valores_novos = valores_normalizados.map(mapa)

    # Valores sem regra permanecem exatamente como estavam
    valores_novos = valores_novos.fillna(valores_antes)

    # Como as novas categorias são numéricas, tentamos manter
    # a coluna numérica quando isso for possível.
    convertido_numerico = pd.to_numeric(
        valores_novos,
        errors="coerce"
    )

    if convertido_numerico.notna().all():

        valores_novos = convertido_numerico


    df[coluna_real] = valores_novos


    # --------------------------------------------------------
    # Contar alterações reais
    # --------------------------------------------------------

    alteracoes = (
        valores_antes.astype("string")
        != df[coluna_real].astype("string")
    ).sum()

    alteracoes = int(alteracoes)

    total_correspondencias += correspondencias
    total_alteracoes += alteracoes
    colunas_processadas += 1


    # --------------------------------------------------------
    # Relatório
    # --------------------------------------------------------

    status = "ALTERADO"

    if correspondencias == 0:
        status = "NENHUMA CORRESPONDÊNCIA"

    relatorio.append({
        "codigo_excel": codigo,
        "coluna_parquet": coluna_real,
        "regra": regra,
        "status": status,
        "correspondencias": correspondencias,
        "alteracoes": alteracoes
    })


    # --------------------------------------------------------
    # Mostrar progresso
    # --------------------------------------------------------

    print(
        f"{status} | {codigo} | "
        f"correspondências: {correspondencias:,} | "
        f"alterações: {alteracoes:,}"
    )


# ============================================================
# 9. SALVAR O NOVO PARQUET
# ============================================================

df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="snappy",
    index=False
)


# ============================================================
# 10. RELATÓRIOS
# ============================================================

relatorio_df = pd.DataFrame(relatorio)
diagnostico_df = pd.DataFrame(diagnostico)


# Relatório geral
relatorio_df.to_excel(
    "05_RELATORIO_RECODIFICACAO.xlsx",
    index=False
)

# Diagnóstico detalhado:
# mostra exatamente quais valores foram encontrados
# e para qual valor cada um foi recodificado.
diagnostico_df.to_excel(
    "05_DIAGNOSTICO_RECODIFICACAO.xlsx",
    index=False
)


# ============================================================
# 11. RESULTADO FINAL
# ============================================================

print()
print("=" * 80)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

print(
    f"Colunas processadas:          {colunas_processadas}"
)

print(
    f"Total de correspondências:    {total_correspondencias:,}"
)

print(
    f"Total de valores alterados:   {total_alteracoes:,}"
)

print(
    f"Arquivo criado:               {arquivo_saida}"
)

print(
    "Relatório geral:              "
    "05_RELATORIO_RECODIFICACAO.xlsx"
)

print(
    "Diagnóstico detalhado:        "
    "05_DIAGNOSTICO_RECODIFICACAO.xlsx"
)

print()
print("=" * 80)
print("RELATÓRIO FINAL")
print("=" * 80)

display(relatorio_df)

print()
print("=" * 80)
print("DIAGNÓSTICO DOS VALORES ENCONTRADOS")
print("=" * 80)

display(diagnostico_df)


TIPOS DAS COLUNAS DO PARQUET
REGIAO - código de regiões                                                                                                                                                                                                                                   int64
UF - código UF                                                                                                                                                                                                                                               int64
CAPITAL - indica se é município da capital                                                                                                                                                                                                                   int64
B01001A - Qual é o seu sexo?                                                                                                                                                                      

,codigo_excel,coluna_parquet,regra,status,correspondencias,alteracoes
0,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...","1=3,2=1,-9/-2/-1=0",ALTERADO,148736,148736
1,E01P117,"E01P117 - NOS ÚLTIMOS 12 MESES, a escola conto...","1=4,2=2,3=3,4=1,-9/-2/-1=0",ALTERADO,148736,148736
2,E01P11801,E01P11801 - Quais tipos de suporte a escola of...,"1=3,0=1,-9/-2/-1=0)",ALTERADO,148736,148736
3,E01P11802,E01P11802 - Quais tipos de suporte a escola of...,"1=3,0=1,-9/-2/-1=0)",ALTERADO,148736,148736
4,E01P8505,"E01P8505 - NOS ÚLTIMOS 12 MESES, a escola dese...","1=3,0=1,-9/-2/-1=0)",ALTERADO,148736,148736
5,E01P8510,"E01P8510 - NOS ÚLTIMOS 12 MESES, a escola dese...","1=3,0=1,-9/-2/-1=0)",ALTERADO,148736,148736
6,B07006,"B07006 - NOS ÚLTIMOS 30 DIAS, com que frequênc...","1/2=1,3=2,4/5=3,-9/-2/-1=0",ALTERADO,148736,148736
7,B07007B,"B07007B - NOS ÚLTIMOS 30 DIAS, quantas vezes a...","1=1,2/3/4=3,-9/-2/-1=0",ALTERADO,148736,148736
8,B07009,"B07009 - NOS ÚLTIMOS 30 DIAS, você esculachou,...","1=3,2=1,-9/-2/-2=0",ALTERADO,148736,148736
9,B07011A,"B07011A - NOS ÚLTIMOS 30 DIAS, quantas vezes a...","1=1,2/3/4=3,-9/-2/-1=0",ALTERADO,148736,148736



DIAGNÓSTICO DOS VALORES ENCONTRADOS


,codigo_excel,coluna_parquet,valor_encontrado,quantidade,novo_valor
0,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",2,99285,1
1,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",1,48565,3
2,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",-9,527,0
3,B12009,"B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você...",-2,359,0
4,E01P117,"E01P117 - NOS ÚLTIMOS 12 MESES, a escola conto...",1,76050,4
...,...,...,...,...,...
218,B05002A,B05002A - Alguma vez na vida você tomou um cop...,1,65922,3
219,B05002A,B05002A - Alguma vez na vida você tomou um cop...,2,58661,1
220,B05002A,B05002A - Alguma vez na vida você tomou um cop...,-1,23832,0
221,B05002A,B05002A - Alguma vez na vida você tomou um cop...,-9,165,0
